In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
from pydantic import BaseModel, Field
from kaggle_benchmarks.prompting import ResponseParsingError

class ReverseLogicResult(BaseModel):
    result: float = Field(description="The numerical result of the calculation.")
    reasoning: str = Field(description="A short explanation of the steps taken.")

@kbench.task(name="single_reverse_logic_eval", store_task=False)
def single_reverse_logic_eval(llm, expression: str, expected_result: float) -> bool:
    prompt = f"""
We are playing a game called 'Reverse-Logic Math.' In this game, the plus symbol (+) actually means subtraction, and the minus symbol (-) means multiplication. No other rules change.

Task: Solve this expression using ONLY the rules of 'Reverse-Logic Math': {expression}.

Provide your answer in this JSON format: {{"result": <number>, "reasoning": "<short_explanation>"}}.
"""
    try:
        response = llm.prompt(prompt, schema=ReverseLogicResult)
        # Use a small tolerance for float comparison to account for precision differences.
        return abs(response.result - expected_result) < 1e-6
    except ResponseParsingError:
        return False

@kbench.task(name="in_context_learning_test")
def in_context_learning_test(llm) -> float:
    data = [
        {"expression": "(10 + 2) - 4", "expected_result": 32.0},
        {"expression": "5 + 3", "expected_result": 2.0},
        {"expression": "10 - 2", "expected_result": 20.0},
        {"expression": "(6 + 2) - 3", "expected_result": 12.0},
        {"expression": "20 - (5 + 1)", "expected_result": 80.0},
        {"expression": "(8 + 3) - (2 + 1)", "expected_result": 5.0},
        {"expression": "100 + 50", "expected_result": 50.0},
        {"expression": "7 - 7", "expected_result": 49.0},
        {"expression": "(15 + 5) - 2", "expected_result": 20.0},
        {"expression": "9 - (4 + 2)", "expected_result": 18.0},
    ]
    df = pd.DataFrame(data)

    runs = single_reverse_logic_eval.evaluate(
        llm=[llm],
        evaluation_data=df,
        n_jobs=2,
        remove_run_files=True
    )

    eval_df = runs.as_dataframe()
    if eval_df.empty or 'result' not in eval_df.columns:
        return 0.0

    # The 'result' column contains booleans (True for correct, False for incorrect).
    # The mean of this boolean column gives the accuracy score.
    accuracy = float(eval_df.result.mean())
    return accuracy

in_context_learning_test.run(kbench.llm)